In [1]:
"""
Reviewer analysis for the local quadratic pseudo-Boolean surrogate.

This script uses the first-order nonlinear consensus example and reports:
  1. Training fitting error.
  2. Independent validation error.
  3. Rank correlation and pairwise ranking accuracy.
  4. Top-k candidate overlap and recall.
  5. Whether exact nonlinear re-evaluation changes the surrogate selection.
  6. Screening regret relative to the best true-cost point in the evaluated pool.
  7. Matrix-rank/conditioning diagnostics and underdetermination checks.
  8. Closed-loop comparison between surrogate screening and an oracle-pool
     selection that evaluates every point in the same comparison pool.

The oracle-pool comparison is not a global continuous optimum. It is a controlled
counterfactual that isolates the loss caused by surrogate ranking within the same
finite encoded comparison pool.
"""

from __future__ import annotations

import json
import math
import time
from dataclasses import dataclass, asdict
from itertools import product
from pathlib import Path
from typing import Dict, Iterable, List, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp
from scipy.stats import kendalltau, spearmanr


# ============================================================
# OUTPUT AND REPRODUCIBILITY
# ============================================================

OUTPUT_DIR = Path("first_order_surrogate_analysis")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GLOBAL_SEED = 20260721
BH_SEED_BASE = 1000
POOL_SEED_BASE = 5000

# Run the paired closed-loop counterfactual.
RUN_ORACLE_COUNTERFACTUAL = True

# If the encoded domain has at most this many bits, evaluate every bitstring
# not used for training. Otherwise, use an independent random validation pool.
EXHAUSTIVE_VALIDATION_MAX_QUBITS = 12
VALIDATION_POOL_SIZE = 512

# Number of surrogate-ranked candidates retained for exact nonlinear evaluation.
SCREEN_TOP_K = 32

# Pair count used for empirical pairwise ranking accuracy.
PAIRWISE_COMPARISONS = 10000

# Small denominator used in normalized and relative quantities.
EPS = 1e-12


# ============================================================
# FIRST-ORDER CONSENSUS CONFIGURATION
# ============================================================

N_AGENTS = 5

A = np.array(
    [
        [0, 1, 0, 0, 1],
        [1, 0, 1, 0, 0],
        [0, 1, 0, 1, 0],
        [0, 0, 1, 0, 1],
        [1, 0, 0, 1, 0],
    ],
    dtype=float,
)
D = np.diag(A.sum(axis=1))
L = D - A

X0_GLOBAL = np.array([2.0, -2.5, 3.8, -3.2, 0.3], dtype=float)

T_HORIZON_COST = 0.25
N_TIME_COST = 150
W_Z = 1.0
W_U = 0.1
W_LYAP = 1.0

PARAM_NAMES = ["alpha", "beta", "k", "theta2", "theta4"]
P_MIN_GLOBAL = np.array([0.0, 0.0, 0.0, 0.0, 0.0])
P_MAX_GLOBAL = np.array([50.0, 2.0, 50.0, 25.0, 25.0])

BH_POP_SIZE = 20
BH_MAX_ITERS = 200
BH_FREEZE_WIDTH = 5.0

T_TOTAL = 10.0
DT_DECISION = 0.25
CONS_TOL = 1e-8

TRAIN_SAMPLE_FACTOR = 4
MIN_TRAIN_SAMPLES = 64

BIT_WIDTH_THRESHOLDS = [5.0, 20.0]
BIT_ALLOCATION = [2, 3, 4]
MAX_BITS_PER_PARAM = 4


# ============================================================
# CLOSED-LOOP MODEL AND TRUE NONLINEAR COST
# ============================================================


def closed_loop_dynamics(
    t: float,
    x: np.ndarray,
    alpha: float,
    beta: float,
    k_gain: float,
) -> np.ndarray:
    del t
    x = np.asarray(x, dtype=float)
    local_drift = x + x**3
    control_u = -alpha * x - beta * x**3 - k_gain * (L @ x)
    return local_drift + control_u


def control_law(
    x: np.ndarray,
    alpha: float,
    beta: float,
    k_gain: float,
) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    return -alpha * x - beta * x**3 - k_gain * (L @ x)


def consensus_disagreement(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    if x.ndim == 1:
        return x - np.mean(x)
    if x.ndim == 2:
        return x - np.mean(x, axis=0, keepdims=True)
    raise ValueError("x must have shape (N,) or (N,T).")


def disagreement_dynamics(dxdt: np.ndarray) -> np.ndarray:
    dxdt = np.asarray(dxdt, dtype=float)
    if dxdt.ndim == 1:
        return dxdt - np.mean(dxdt)
    if dxdt.ndim == 2:
        return dxdt - np.mean(dxdt, axis=0, keepdims=True)
    raise ValueError("dxdt must have shape (N,) or (N,T).")


def lyapunov_dvdt(
    x: np.ndarray,
    dxdt: np.ndarray,
    theta2: float,
    theta4: float,
) -> float:
    e = consensus_disagreement(x)
    de_dt = disagreement_dynamics(dxdt)
    d_v_de = theta2 * e + theta4 * e**3
    return float(np.dot(d_v_de, de_dt))


def simulate_horizon_cost(
    x0: np.ndarray,
    alpha: float,
    beta: float,
    k_gain: float,
    theta2: float,
    theta4: float,
    t_horizon: float | None = None,
    n_time: int | None = None,
    w_z: float | None = None,
    w_u: float | None = None,
    w_lyap: float | None = None,
) -> float:
    """Evaluate the original nonlinear short-horizon objective."""
    t_horizon = T_HORIZON_COST if t_horizon is None else float(t_horizon)
    n_time = N_TIME_COST if n_time is None else int(n_time)
    w_z = W_Z if w_z is None else float(w_z)
    w_u = W_U if w_u is None else float(w_u)
    w_lyap = W_LYAP if w_lyap is None else float(w_lyap)

    t_eval = np.linspace(0.0, t_horizon, n_time)

    def dyn(t: float, x: np.ndarray) -> np.ndarray:
        return closed_loop_dynamics(t, x, alpha, beta, k_gain)

    sol = solve_ivp(
        dyn,
        (0.0, t_horizon),
        np.asarray(x0, dtype=float),
        t_eval=t_eval,
        method="RK45",
        rtol=1e-6,
        atol=1e-8,
    )

    if not sol.success or np.any(~np.isfinite(sol.y)):
        return 1e6
    if np.any(np.abs(sol.y) > 1e3):
        return 1e6

    t = sol.t
    x_trajectory = sol.y
    z = L @ x_trajectory
    u = control_law(x_trajectory, alpha, beta, k_gain)

    performance_integrand = w_z * np.sum(z**2, axis=0) + w_u * np.sum(u**2, axis=0)

    dv_values = np.empty(t.size, dtype=float)
    for idx, tt in enumerate(t):
        x_t = x_trajectory[:, idx]
        dx_t = dyn(tt, x_t)
        dv_values[idx] = lyapunov_dvdt(x_t, dx_t, theta2, theta4)

    violation_integrand = np.maximum(0.0, dv_values) ** 2
    j_perf = float(np.trapz(performance_integrand, t))
    j_lyap = float(np.trapz(violation_integrand, t))
    return j_perf + w_lyap * j_lyap


# ============================================================
# BLACK-HOLE CALIBRATION
# ============================================================


def evaluate_parameter_vector(x0: np.ndarray, p: Sequence[float]) -> float:
    return simulate_horizon_cost(x0, *map(float, p))


def bh_range_calibration(
    p_min_init: np.ndarray,
    p_max_init: np.ndarray,
    x0: np.ndarray,
    seed: int,
) -> Tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    n_params = len(PARAM_NAMES)

    p_min = np.asarray(p_min_init, dtype=float).copy()
    p_max = np.asarray(p_max_init, dtype=float).copy()
    stars = rng.uniform(p_min, p_max, size=(BH_POP_SIZE, n_params))
    active = np.ones(n_params, dtype=bool)

    for _ in range(BH_MAX_ITERS):
        costs = np.array([evaluate_parameter_vector(x0, p) for p in stars])
        best_idx = int(np.argmin(costs))
        black_hole = stars[best_idx].copy()

        active_indices = np.flatnonzero(active)
        if active_indices.size:
            for i in range(BH_POP_SIZE):
                if i == best_idx:
                    continue
                rand_vec = rng.random(active_indices.size)
                stars[i, active_indices] += rand_vec * (
                    black_hole[active_indices] - stars[i, active_indices]
                )

        stars = np.clip(stars, p_min, p_max)

        for j in active_indices:
            p_min[j] = max(float(np.min(stars[:, j])), P_MIN_GLOBAL[j])
            p_max[j] = min(float(np.max(stars[:, j])), P_MAX_GLOBAL[j])
            if p_max[j] - p_min[j] <= BH_FREEZE_WIDTH:
                active[j] = False

        if not np.any(active):
            break

    return p_min, p_max


# ============================================================
# BINARY ENCODING AND QUADRATIC BASIS
# ============================================================


def choose_bits_for_width(width: float) -> int:
    if width <= BIT_WIDTH_THRESHOLDS[0]:
        return min(BIT_ALLOCATION[0], MAX_BITS_PER_PARAM)
    if width <= BIT_WIDTH_THRESHOLDS[1]:
        return min(BIT_ALLOCATION[1], MAX_BITS_PER_PARAM)
    return min(BIT_ALLOCATION[2], MAX_BITS_PER_PARAM)


def allocate_bits_for_parameters(
    p_min: np.ndarray,
    p_max: np.ndarray,
) -> Tuple[List[int], int]:
    widths = np.asarray(p_max) - np.asarray(p_min)
    bits = [choose_bits_for_width(float(width)) for width in widths]
    return bits, int(sum(bits))


def decode_bitstring_to_params(
    bitstring: str,
    p_min: np.ndarray,
    p_max: np.ndarray,
    bits_per_param: Sequence[int],
) -> np.ndarray:
    if len(bitstring) != int(sum(bits_per_param)):
        raise ValueError("Bitstring length does not match the bit allocation.")

    decoded: List[float] = []
    index = 0
    for parameter_index, n_bits in enumerate(bits_per_param):
        substring = bitstring[index : index + n_bits]
        index += n_bits
        integer_value = int(substring, 2)
        levels = 2**n_bits - 1
        value = p_min[parameter_index] + (
            p_max[parameter_index] - p_min[parameter_index]
        ) * integer_value / levels
        decoded.append(float(value))
    return np.asarray(decoded, dtype=float)


def build_diagonal_pauli_basis(num_qubits: int) -> List[str]:
    labels = ["I" * num_qubits]
    for i in range(num_qubits):
        label = ["I"] * num_qubits
        label[i] = "Z"
        labels.append("".join(label))
    for i in range(num_qubits):
        for j in range(i + 1, num_qubits):
            label = ["I"] * num_qubits
            label[i] = "Z"
            label[j] = "Z"
            labels.append("".join(label))
    return labels


def eigenvalue_of_pauli_on_bitstring(pauli_label: str, bitstring: str) -> float:
    eigenvalue = 1.0
    for pauli, bit in zip(pauli_label, bitstring):
        if pauli == "Z":
            eigenvalue *= 1.0 if bit == "0" else -1.0
    return eigenvalue


def feature_matrix(bitstrings: Sequence[str], paulis: Sequence[str]) -> np.ndarray:
    matrix = np.empty((len(bitstrings), len(paulis)), dtype=float)
    for row, bitstring in enumerate(bitstrings):
        matrix[row, :] = [
            eigenvalue_of_pauli_on_bitstring(label, bitstring) for label in paulis
        ]
    return matrix


def predict_surrogate(
    bitstrings: Sequence[str],
    paulis: Sequence[str],
    coefficients: np.ndarray,
) -> np.ndarray:
    return feature_matrix(bitstrings, paulis) @ coefficients


def all_bitstrings(num_qubits: int) -> List[str]:
    return ["".join(bits) for bits in product("01", repeat=num_qubits)]


def sample_unique_bitstrings(
    num_qubits: int,
    count: int,
    rng: np.random.Generator,
    excluded: Iterable[str] | None = None,
) -> List[str]:
    excluded_set = set() if excluded is None else set(excluded)
    available_count = 2**num_qubits - len(excluded_set)
    count = min(int(count), available_count)

    if num_qubits <= EXHAUSTIVE_VALIDATION_MAX_QUBITS and available_count <= 50000:
        candidates = [s for s in all_bitstrings(num_qubits) if s not in excluded_set]
        if count == len(candidates):
            return candidates
        indices = rng.choice(len(candidates), size=count, replace=False)
        return [candidates[int(i)] for i in indices]

    selected: set[str] = set()
    while len(selected) < count:
        candidate = "".join(rng.choice(["0", "1"], size=num_qubits))
        if candidate not in excluded_set:
            selected.add(candidate)
    return sorted(selected)


def evaluate_true_costs(
    bitstrings: Sequence[str],
    x0: np.ndarray,
    p_min: np.ndarray,
    p_max: np.ndarray,
    bits_per_param: Sequence[int],
) -> np.ndarray:
    values = np.empty(len(bitstrings), dtype=float)
    for idx, bitstring in enumerate(bitstrings):
        parameters = decode_bitstring_to_params(
            bitstring, p_min, p_max, bits_per_param
        )
        values[idx] = evaluate_parameter_vector(x0, parameters)
    return values


# ============================================================
# SURROGATE METRICS
# ============================================================


def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray, prefix: str) -> Dict[str, float]:
    residual = np.asarray(y_pred) - np.asarray(y_true)
    rmse = float(np.sqrt(np.mean(residual**2)))
    mae = float(np.mean(np.abs(residual)))
    max_ae = float(np.max(np.abs(residual)))
    target_range = float(np.max(y_true) - np.min(y_true))
    target_mean_abs = float(np.mean(np.abs(y_true)))
    ss_res = float(np.sum(residual**2))
    ss_tot = float(np.sum((y_true - np.mean(y_true)) ** 2))
    r2 = 1.0 - ss_res / ss_tot if ss_tot > EPS else float("nan")

    return {
        f"{prefix}_rmse": rmse,
        f"{prefix}_mae": mae,
        f"{prefix}_max_abs_error": max_ae,
        f"{prefix}_nrmse_range": rmse / max(target_range, EPS),
        f"{prefix}_nrmse_mean_abs": rmse / max(target_mean_abs, EPS),
        f"{prefix}_r2": r2,
    }


def empirical_pairwise_ranking_accuracy(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    pair_count: int,
    rng: np.random.Generator,
) -> float:
    n = len(y_true)
    if n < 2:
        return float("nan")

    correct = 0
    valid = 0
    for _ in range(pair_count):
        i, j = rng.choice(n, size=2, replace=False)
        true_difference = y_true[i] - y_true[j]
        pred_difference = y_pred[i] - y_pred[j]
        if abs(true_difference) <= EPS:
            continue
        valid += 1
        if np.sign(true_difference) == np.sign(pred_difference):
            correct += 1
    return correct / valid if valid else float("nan")


def safe_spearman(y_true: np.ndarray, y_pred: np.ndarray) -> Tuple[float, float]:
    result = spearmanr(y_true, y_pred)
    return float(result.statistic), float(result.pvalue)


def safe_kendall(y_true: np.ndarray, y_pred: np.ndarray) -> Tuple[float, float]:
    result = kendalltau(y_true, y_pred)
    return float(result.statistic), float(result.pvalue)


def top_k_overlap_fraction(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    k: int,
) -> float:
    k = min(k, len(y_true))
    true_top = set(np.argsort(y_true)[:k].tolist())
    pred_top = set(np.argsort(y_pred)[:k].tolist())
    return len(true_top & pred_top) / max(k, 1)


def one_based_rank(values: np.ndarray, index: int) -> int:
    order = np.argsort(values)
    return int(np.flatnonzero(order == index)[0] + 1)


@dataclass
class EpochAnalysis:
    decision_step: int
    decision_time: float
    num_qubits: int
    num_basis_terms: int
    encoded_domain_size: int
    train_samples: int
    validation_samples: int
    design_matrix_rank: int
    design_matrix_condition: float
    is_underdetermined: bool
    train_rmse: float
    train_mae: float
    train_max_abs_error: float
    train_nrmse_range: float
    train_nrmse_mean_abs: float
    train_r2: float
    validation_rmse: float
    validation_mae: float
    validation_max_abs_error: float
    validation_nrmse_range: float
    validation_nrmse_mean_abs: float
    validation_r2: float
    spearman_rho: float
    spearman_pvalue: float
    kendall_tau: float
    kendall_pvalue: float
    pairwise_ranking_accuracy: float
    top1_overlap: float
    top5_overlap: float
    top10_overlap: float
    top32_overlap: float
    true_best_in_surrogate_topk: bool
    surrogate_best_true_rank: int
    true_best_surrogate_rank: int
    re_evaluation_changed_selection: bool
    surrogate_rank1_true_cost: float
    re_evaluated_topk_true_cost: float
    oracle_pool_true_cost: float
    screening_regret_absolute: float
    screening_regret_relative: float
    selected_bitstring: str
    surrogate_rank1_bitstring: str
    oracle_pool_bitstring: str
    fit_seconds: float
    validation_seconds: float


@dataclass
class FittedSurrogate:
    paulis: List[str]
    coefficients: np.ndarray
    train_bitstrings: List[str]
    train_true_costs: np.ndarray
    train_predictions: np.ndarray
    validation_bitstrings: List[str]
    validation_true_costs: np.ndarray
    validation_predictions: np.ndarray
    analysis: EpochAnalysis


def fit_and_analyze_surrogate(
    x0: np.ndarray,
    p_min: np.ndarray,
    p_max: np.ndarray,
    bits_per_param: Sequence[int],
    decision_step: int,
    decision_time: float,
    seed: int,
) -> FittedSurrogate:
    rng = np.random.default_rng(seed)
    num_qubits = int(sum(bits_per_param))
    paulis = build_diagonal_pauli_basis(num_qubits)
    num_basis = len(paulis)
    domain_size = 2**num_qubits

    requested_train = max(TRAIN_SAMPLE_FACTOR * num_basis, MIN_TRAIN_SAMPLES)
    train_count = min(requested_train, domain_size)
    train_bitstrings = sample_unique_bitstrings(
        num_qubits, train_count, rng, excluded=None
    )

    fit_start = time.perf_counter()
    train_matrix = feature_matrix(train_bitstrings, paulis)
    train_true = evaluate_true_costs(
        train_bitstrings, x0, p_min, p_max, bits_per_param
    )
    coefficients, *_ = np.linalg.lstsq(train_matrix, train_true, rcond=None)
    train_predictions = train_matrix @ coefficients
    fit_seconds = time.perf_counter() - fit_start

    excluded = set(train_bitstrings)
    remaining_count = domain_size - len(excluded)
    if num_qubits <= EXHAUSTIVE_VALIDATION_MAX_QUBITS:
        validation_count = remaining_count
    else:
        validation_count = min(VALIDATION_POOL_SIZE, remaining_count)

    validation_start = time.perf_counter()
    validation_bitstrings = sample_unique_bitstrings(
        num_qubits, validation_count, rng, excluded=excluded
    )
    validation_true = evaluate_true_costs(
        validation_bitstrings, x0, p_min, p_max, bits_per_param
    )
    validation_predictions = predict_surrogate(
        validation_bitstrings, paulis, coefficients
    )
    validation_seconds = time.perf_counter() - validation_start

    metrics: Dict[str, float] = {}
    metrics.update(regression_metrics(train_true, train_predictions, "train"))
    metrics.update(
        regression_metrics(validation_true, validation_predictions, "validation")
    )

    spearman_rho, spearman_p = safe_spearman(validation_true, validation_predictions)
    kendall_tau, kendall_p = safe_kendall(validation_true, validation_predictions)
    pairwise_accuracy = empirical_pairwise_ranking_accuracy(
        validation_true,
        validation_predictions,
        min(PAIRWISE_COMPARISONS, max(len(validation_true) * 10, 100)),
        rng,
    )

    surrogate_order = np.argsort(validation_predictions)
    true_order = np.argsort(validation_true)
    top_k = min(SCREEN_TOP_K, len(validation_bitstrings))
    surrogate_top_k = surrogate_order[:top_k]

    surrogate_rank1_index = int(surrogate_order[0])
    true_best_index = int(true_order[0])
    reevaluated_index = int(
        surrogate_top_k[np.argmin(validation_true[surrogate_top_k])]
    )

    surrogate_rank1_true_cost = float(validation_true[surrogate_rank1_index])
    reevaluated_true_cost = float(validation_true[reevaluated_index])
    oracle_true_cost = float(validation_true[true_best_index])
    regret_abs = reevaluated_true_cost - oracle_true_cost
    regret_rel = regret_abs / max(abs(oracle_true_cost), EPS)

    singular_values = np.linalg.svd(train_matrix, compute_uv=False)
    condition = (
        float(singular_values[0] / singular_values[-1])
        if singular_values[-1] > EPS
        else float("inf")
    )

    analysis = EpochAnalysis(
        decision_step=decision_step,
        decision_time=decision_time,
        num_qubits=num_qubits,
        num_basis_terms=num_basis,
        encoded_domain_size=domain_size,
        train_samples=len(train_bitstrings),
        validation_samples=len(validation_bitstrings),
        design_matrix_rank=int(np.linalg.matrix_rank(train_matrix)),
        design_matrix_condition=condition,
        is_underdetermined=len(train_bitstrings) < num_basis,
        spearman_rho=spearman_rho,
        spearman_pvalue=spearman_p,
        kendall_tau=kendall_tau,
        kendall_pvalue=kendall_p,
        pairwise_ranking_accuracy=pairwise_accuracy,
        top1_overlap=top_k_overlap_fraction(validation_true, validation_predictions, 1),
        top5_overlap=top_k_overlap_fraction(validation_true, validation_predictions, 5),
        top10_overlap=top_k_overlap_fraction(validation_true, validation_predictions, 10),
        top32_overlap=top_k_overlap_fraction(validation_true, validation_predictions, 32),
        true_best_in_surrogate_topk=bool(true_best_index in surrogate_top_k),
        surrogate_best_true_rank=one_based_rank(validation_true, surrogate_rank1_index),
        true_best_surrogate_rank=one_based_rank(validation_predictions, true_best_index),
        re_evaluation_changed_selection=bool(reevaluated_index != surrogate_rank1_index),
        surrogate_rank1_true_cost=surrogate_rank1_true_cost,
        re_evaluated_topk_true_cost=reevaluated_true_cost,
        oracle_pool_true_cost=oracle_true_cost,
        screening_regret_absolute=float(regret_abs),
        screening_regret_relative=float(regret_rel),
        selected_bitstring=validation_bitstrings[reevaluated_index],
        surrogate_rank1_bitstring=validation_bitstrings[surrogate_rank1_index],
        oracle_pool_bitstring=validation_bitstrings[true_best_index],
        fit_seconds=float(fit_seconds),
        validation_seconds=float(validation_seconds),
        **metrics,
    )

    return FittedSurrogate(
        paulis=paulis,
        coefficients=coefficients,
        train_bitstrings=train_bitstrings,
        train_true_costs=train_true,
        train_predictions=train_predictions,
        validation_bitstrings=validation_bitstrings,
        validation_true_costs=validation_true,
        validation_predictions=validation_predictions,
        analysis=analysis,
    )


# ============================================================
# CLOSED-LOOP EXPERIMENT
# ============================================================


@dataclass
class ClosedLoopSummary:
    mode: str
    completed_steps: int
    final_time: float
    final_consensus_error: float
    rms_consensus_error_at_epochs: float
    maximum_consensus_error_at_epochs: float
    cumulative_selected_short_horizon_cost: float
    mean_selected_short_horizon_cost: float


def integrate_one_decision_interval(
    x0: np.ndarray,
    parameters: Sequence[float],
) -> np.ndarray:
    alpha, beta, k_gain, _, _ = map(float, parameters)

    def dyn(t: float, x: np.ndarray) -> np.ndarray:
        return closed_loop_dynamics(t, x, alpha, beta, k_gain)

    sol = solve_ivp(
        dyn,
        (0.0, DT_DECISION),
        x0,
        t_eval=np.linspace(0.0, DT_DECISION, 200),
        method="RK45",
        rtol=1e-6,
        atol=1e-8,
    )
    if not sol.success:
        raise RuntimeError(sol.message)
    return sol.y[:, -1].copy()


def run_closed_loop_experiment(
    mode: str,
    collect_surrogate_reports: bool,
) -> Tuple[ClosedLoopSummary, pd.DataFrame, pd.DataFrame]:
    if mode not in {"surrogate_screening", "oracle_pool"}:
        raise ValueError("Unknown selection mode.")

    x_current = X0_GLOBAL.copy()
    p_min_current = P_MIN_GLOBAL.copy()
    p_max_current = P_MAX_GLOBAL.copy()

    epoch_rows: List[Dict[str, object]] = []
    selected_rows: List[Dict[str, object]] = []
    consensus_errors: List[float] = []
    selected_costs: List[float] = []

    max_steps = int(T_TOTAL / DT_DECISION)

    for step in range(max_steps):
        decision_time = step * DT_DECISION
        p_min_current, p_max_current = bh_range_calibration(
            p_min_current,
            p_max_current,
            x_current,
            seed=BH_SEED_BASE + step,
        )
        bits_per_param, num_qubits = allocate_bits_for_parameters(
            p_min_current, p_max_current
        )

        fitted = fit_and_analyze_surrogate(
            x0=x_current,
            p_min=p_min_current,
            p_max=p_max_current,
            bits_per_param=bits_per_param,
            decision_step=step + 1,
            decision_time=decision_time,
            seed=POOL_SEED_BASE + step,
        )

        if collect_surrogate_reports:
            epoch_rows.append(asdict(fitted.analysis))

            epoch_point_df = pd.DataFrame(
                {
                    "decision_step": step + 1,
                    "bitstring": fitted.validation_bitstrings,
                    "true_cost": fitted.validation_true_costs,
                    "surrogate_cost": fitted.validation_predictions,
                    "true_rank": pd.Series(fitted.validation_true_costs).rank(
                        method="min", ascending=True
                    ).astype(int),
                    "surrogate_rank": pd.Series(fitted.validation_predictions).rank(
                        method="min", ascending=True
                    ).astype(int),
                }
            )
            epoch_point_df.to_csv(
                OUTPUT_DIR / f"epoch_{step + 1:02d}_validation_points.csv",
                index=False,
            )

        if mode == "surrogate_screening":
            selected_bitstring = fitted.analysis.selected_bitstring
        else:
            selected_bitstring = fitted.analysis.oracle_pool_bitstring

        selected_index = fitted.validation_bitstrings.index(selected_bitstring)
        selected_true_cost = float(fitted.validation_true_costs[selected_index])
        parameters = decode_bitstring_to_params(
            selected_bitstring,
            p_min_current,
            p_max_current,
            bits_per_param,
        )

        selected_rows.append(
            {
                "mode": mode,
                "decision_step": step + 1,
                "decision_time": decision_time,
                "num_qubits": num_qubits,
                "bits_per_param": json.dumps(bits_per_param),
                "selected_bitstring": selected_bitstring,
                "selected_true_cost": selected_true_cost,
                **{name: value for name, value in zip(PARAM_NAMES, parameters)},
            }
        )
        selected_costs.append(selected_true_cost)

        x_current = integrate_one_decision_interval(x_current, parameters)
        consensus_error = float(np.linalg.norm(L @ x_current, 2))
        consensus_errors.append(consensus_error)

        print(
            f"[{mode}] step={step + 1:02d}, nq={num_qubits:2d}, "
            f"J={selected_true_cost:.6e}, ||Lx||={consensus_error:.3e}"
        )

        if consensus_error <= CONS_TOL:
            break

    summary = ClosedLoopSummary(
        mode=mode,
        completed_steps=len(consensus_errors),
        final_time=len(consensus_errors) * DT_DECISION,
        final_consensus_error=float(consensus_errors[-1]),
        rms_consensus_error_at_epochs=float(
            np.sqrt(np.mean(np.asarray(consensus_errors) ** 2))
        ),
        maximum_consensus_error_at_epochs=float(np.max(consensus_errors)),
        cumulative_selected_short_horizon_cost=float(np.sum(selected_costs)),
        mean_selected_short_horizon_cost=float(np.mean(selected_costs)),
    )

    return summary, pd.DataFrame(epoch_rows), pd.DataFrame(selected_rows)


# ============================================================
# REPORTING AND FIGURES
# ============================================================


def save_summary_tables(epoch_df: pd.DataFrame) -> None:
    numeric_columns = epoch_df.select_dtypes(include=[np.number]).columns
    aggregate_rows = []
    for column in numeric_columns:
        aggregate_rows.append(
            {
                "metric": column,
                "mean": epoch_df[column].mean(),
                "std": epoch_df[column].std(ddof=0),
                "min": epoch_df[column].min(),
                "max": epoch_df[column].max(),
            }
        )
    pd.DataFrame(aggregate_rows).to_csv(
        OUTPUT_DIR / "surrogate_metrics_summary.csv", index=False
    )

    key_columns = [
        "decision_step",
        "num_qubits",
        "train_samples",
        "validation_samples",
        "design_matrix_rank",
        "num_basis_terms",
        "is_underdetermined",
        "train_nrmse_range",
        "validation_nrmse_range",
        "validation_r2",
        "spearman_rho",
        "kendall_tau",
        "pairwise_ranking_accuracy",
        "top10_overlap",
        "top32_overlap",
        "true_best_in_surrogate_topk",
        "re_evaluation_changed_selection",
        "screening_regret_relative",
    ]
    epoch_df[key_columns].to_csv(
        OUTPUT_DIR / "surrogate_key_results.csv", index=False
    )


def plot_surrogate_metrics(epoch_df: pd.DataFrame) -> None:
    plt.rcParams.update(
        {
            "font.family": "serif",
            "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
            "mathtext.fontset": "stix",
            "font.size": 10,
            "axes.labelsize": 10,
            "legend.fontsize": 9,
            "figure.dpi": 180,
            "pdf.fonttype": 42,
        }
    )

    x = epoch_df["decision_step"].to_numpy()

    fig, ax = plt.subplots(figsize=(3.5, 2.5), constrained_layout=True)
    ax.plot(x, epoch_df["train_nrmse_range"], marker="o", label="Training")
    ax.plot(x, epoch_df["validation_nrmse_range"], marker="s", label="Validation")
    ax.set_xlabel("Redesign epoch")
    ax.set_ylabel("Range-normalized RMSE")
    ax.grid(True, linestyle=":")
    ax.legend()
    fig.savefig(OUTPUT_DIR / "surrogate_error_by_epoch.pdf", bbox_inches="tight")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(3.5, 2.5), constrained_layout=True)
    ax.plot(x, epoch_df["spearman_rho"], marker="o", label="Spearman")
    ax.plot(x, epoch_df["kendall_tau"], marker="s", label="Kendall")
    ax.plot(
        x,
        epoch_df["pairwise_ranking_accuracy"],
        marker="^",
        label="Pairwise accuracy",
    )
    ax.set_xlabel("Redesign epoch")
    ax.set_ylabel("Ranking metric")
    ax.set_ylim(-0.05, 1.05)
    ax.grid(True, linestyle=":")
    ax.legend()
    fig.savefig(OUTPUT_DIR / "surrogate_ranking_by_epoch.pdf", bbox_inches="tight")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(3.5, 2.5), constrained_layout=True)
    ax.plot(x, epoch_df["top5_overlap"], marker="o", label="Top-5")
    ax.plot(x, epoch_df["top10_overlap"], marker="s", label="Top-10")
    ax.plot(x, epoch_df["top32_overlap"], marker="^", label="Top-32")
    ax.set_xlabel("Redesign epoch")
    ax.set_ylabel("Top-$k$ overlap")
    ax.set_ylim(-0.05, 1.05)
    ax.grid(True, linestyle=":")
    ax.legend()
    fig.savefig(OUTPUT_DIR / "surrogate_topk_overlap.pdf", bbox_inches="tight")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(3.5, 2.5), constrained_layout=True)
    ax.semilogy(
        x,
        np.maximum(epoch_df["screening_regret_absolute"], 1e-16),
        marker="o",
    )
    ax.set_xlabel("Redesign epoch")
    ax.set_ylabel("Screening regret")
    ax.grid(True, linestyle=":")
    fig.savefig(OUTPUT_DIR / "surrogate_screening_regret.pdf", bbox_inches="tight")
    plt.close(fig)


def print_reviewer_summary(
    epoch_df: pd.DataFrame,
    closed_loop_df: pd.DataFrame,
) -> None:
    print("\n" + "=" * 72)
    print("SURROGATE REVIEWER ANALYSIS SUMMARY")
    print("=" * 72)
    print(f"Redesign epochs analyzed: {len(epoch_df)}")
    print(
        "Validation range-NRMSE: "
        f"mean={epoch_df['validation_nrmse_range'].mean():.4e}, "
        f"max={epoch_df['validation_nrmse_range'].max():.4e}"
    )
    print(
        "Validation R^2: "
        f"mean={epoch_df['validation_r2'].mean():.4f}, "
        f"min={epoch_df['validation_r2'].min():.4f}"
    )
    print(
        "Spearman rank correlation: "
        f"mean={epoch_df['spearman_rho'].mean():.4f}, "
        f"min={epoch_df['spearman_rho'].min():.4f}"
    )
    print(
        "Pairwise ranking accuracy: "
        f"mean={epoch_df['pairwise_ranking_accuracy'].mean():.4f}, "
        f"min={epoch_df['pairwise_ranking_accuracy'].min():.4f}"
    )
    print(
        "True pool-best retained in surrogate top-k: "
        f"{int(epoch_df['true_best_in_surrogate_topk'].sum())}/{len(epoch_df)} epochs"
    )
    print(
        "Exact re-evaluation changed the surrogate rank-1 choice: "
        f"{int(epoch_df['re_evaluation_changed_selection'].sum())}/{len(epoch_df)} epochs"
    )
    print(
        "Relative screening regret: "
        f"mean={epoch_df['screening_regret_relative'].mean():.4e}, "
        f"max={epoch_df['screening_regret_relative'].max():.4e}"
    )
    print(
        "Underdetermined fits: "
        f"{int(epoch_df['is_underdetermined'].sum())}/{len(epoch_df)} epochs"
    )

    print("\nClosed-loop comparison:")
    print(closed_loop_df.to_string(index=False))
    print(f"\nAll files were saved in: {OUTPUT_DIR.resolve()}")


# ============================================================
# MAIN
# ============================================================


def main() -> None:
    surrogate_summary, epoch_df, surrogate_selected_df = run_closed_loop_experiment(
        mode="surrogate_screening",
        collect_surrogate_reports=True,
    )

    if epoch_df.empty:
        raise RuntimeError("No redesign epoch was completed.")

    epoch_df.to_csv(OUTPUT_DIR / "surrogate_metrics_by_epoch.csv", index=False)
    surrogate_selected_df.to_csv(
        OUTPUT_DIR / "surrogate_selected_parameters.csv", index=False
    )
    save_summary_tables(epoch_df)
    plot_surrogate_metrics(epoch_df)

    closed_loop_summaries = [asdict(surrogate_summary)]

    if RUN_ORACLE_COUNTERFACTUAL:
        oracle_summary, _, oracle_selected_df = run_closed_loop_experiment(
            mode="oracle_pool",
            collect_surrogate_reports=False,
        )
        closed_loop_summaries.append(asdict(oracle_summary))
        oracle_selected_df.to_csv(
            OUTPUT_DIR / "oracle_pool_selected_parameters.csv", index=False
        )

    closed_loop_df = pd.DataFrame(closed_loop_summaries)
    closed_loop_df.to_csv(
        OUTPUT_DIR / "closed_loop_surrogate_vs_oracle_pool.csv", index=False
    )

    metadata = {
        "global_seed": GLOBAL_SEED,
        "screen_top_k": SCREEN_TOP_K,
        "validation_pool_size": VALIDATION_POOL_SIZE,
        "exhaustive_validation_max_qubits": EXHAUSTIVE_VALIDATION_MAX_QUBITS,
        "pairwise_comparisons": PAIRWISE_COMPARISONS,
        "run_oracle_counterfactual": RUN_ORACLE_COUNTERFACTUAL,
        "interpretation": (
            "The oracle-pool mode selects the lowest true nonlinear cost in the "
            "same finite validation pool. It is not a global continuous optimum."
        ),
    }
    (OUTPUT_DIR / "analysis_metadata.json").write_text(
        json.dumps(metadata, indent=2), encoding="utf-8"
    )

    print_reviewer_summary(epoch_df, closed_loop_df)


if __name__ == "__main__":
    main()


[surrogate_screening] step=01, nq=10, J=9.360454e+01, ||Lx||=3.433e-03
[surrogate_screening] step=02, nq=10, J=2.011614e-05, ||Lx||=5.684e-06
[surrogate_screening] step=03, nq=10, J=3.120609e-09, ||Lx||=1.051e-08
[surrogate_screening] step=04, nq=10, J=7.328460e-13, ||Lx||=1.226e-10
[oracle_pool] step=01, nq=10, J=9.360454e+01, ||Lx||=3.433e-03
[oracle_pool] step=02, nq=10, J=2.011614e-05, ||Lx||=5.684e-06
[oracle_pool] step=03, nq=10, J=3.120609e-09, ||Lx||=1.051e-08
[oracle_pool] step=04, nq=10, J=7.328460e-13, ||Lx||=1.226e-10

SURROGATE REVIEWER ANALYSIS SUMMARY
Redesign epochs analyzed: 4
Validation range-NRMSE: mean=8.2290e-05, max=3.2664e-04
Validation R^2: mean=1.0000, min=1.0000
Spearman rank correlation: mean=0.9971, min=0.9951
Pairwise ranking accuracy: mean=0.9977, min=0.9930
True pool-best retained in surrogate top-k: 3/4 epochs
Exact re-evaluation changed the surrogate rank-1 choice: 2/4 epochs
Relative screening regret: mean=0.0000e+00, max=0.0000e+00
Underdetermined fit